# 📊 Monitoring MLOps, Santé Opérationnelle, Data Drift & Historique des Optimisations

Ce notebook regroupe l'ensemble du workflow MLOps de production, **augmenté d'un
tracking MLflow systématique** pour répondre aux deux objectifs fixés avec le mentor :

1. **Inspection des dépendances et chargement des artefacts** (modèle XGBoost/Pipeline)
2. **Analyse de la santé opérationnelle de l'API** (volume, taux de succès, latence P95) — *tracée dans MLflow*
3. **Détection du Data Drift & Target Drift** via **Evidently AI** — *rapports historisés (horodatés + artefacts MLflow)*
4. **Benchmark de performance de l'API** (séquentiel + concurrent) — *pour comparer chaque phase d'optimisation*
5. **Diagnostic MLOps et plan d'action**

> ℹ️ **Comment utiliser l'historique** : avant chaque phase d'optimisation, modifie la
> variable `OPTIMIZATION_PHASE_LABEL` (cellule 2) avec une étiquette explicite
> (ex: `"avant_connection_pool"`, puis plus tard `"apres_connection_pool"`), exécute le
> notebook, puis compare les runs dans l'UI MLflow (`mlflow ui`). Chaque exécution
> devient un point d'historique consultable et comparable, y compris le rapport de
> drift HTML complet et le commit Git associé.

## 1. Environment, Imports & Configuration MLflow

In [13]:
import json
import os
import certifi
import subprocess
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import joblib
import pandas as pd
import mlflow
from IPython.display import HTML, display

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset
from evidently.pipeline.column_mapping import ColumnMapping


In [ ]:
# --- Chemins & configuration existants ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOGS_FILE = PROJECT_ROOT / "logs" / "predictions.jsonl"
CONFIG_PRODUCTION_PATH = PROJECT_ROOT / "models" / "config_production.json"
REFERENCE_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "X_train.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CANDIDATE_PATHS = [PROJECT_ROOT / "models" / "best_pipeline_production.joblib"]

with open(PROJECT_ROOT / "models" / "config_production.json", encoding="utf-8") as f:
    config_prod = json.load(f)

FEATURES = config_prod["features_model"]
CATEGORICAL_FEATURES = config_prod["colonnes_cat"]
NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]
SEUIL_PRODUCTION = config_prod["seuil_production"]

In [15]:
# --- Configuration MLflow avec backend SQLite ---
sqlite_db_path = PROJECT_ROOT / "mlflow.db"
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", f"sqlite:///{sqlite_db_path.as_posix()}")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("P8-MLOps-Monitoring")

OPTIMIZATION_PHASE_LABEL = "baseline"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

In [16]:
def get_git_commit() -> str:
    """Récupère le hash court du commit Git courant pour tracer précisément
    quelle version du code a produit ce run de monitoring."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], cwd=str(PROJECT_ROOT)
        ).decode().strip()
    except Exception:
        return "unknown"


def file_hash(path: Path) -> str:
    """Calcule un hash SHA-256 court d'un fichier pour détecter si le jeu de
    données de référence a changé entre deux runs de monitoring."""
    if not path.exists():
        return "missing"
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


print(f"✅ Configuration initialisée. Project Root : {PROJECT_ROOT}")
print(f"✅ MLflow tracking URI : {MLFLOW_TRACKING_URI}")
print(f"✅ Phase d'optimisation étiquetée pour ce run : '{OPTIMIZATION_PHASE_LABEL}'")

✅ Configuration initialisée. Project Root : c:\Users\15GIRAV\Moi\Boulot\OpenClassrooms\P8_Confirmez_vos_competences_en_MLOPS_partie_2_2
✅ MLflow tracking URI : sqlite:///c:/Users/15GIRAV/Moi/Boulot/OpenClassrooms/P8_Confirmez_vos_competences_en_MLOPS_partie_2_2/mlflow.db
✅ Phase d'optimisation étiquetée pour ce run : 'baseline'


## 2. Chargement du Modèle & Extraction des Artefacts

In [17]:
def load_model():
    """Recherche et charge le pipeline entraîné."""
    for model_path in MODEL_CANDIDATE_PATHS:
        if model_path.exists():
            print(f"📍 Chargement du modèle depuis : {model_path}")
            return joblib.load(model_path)
    print("⚠️ Aucun artefact de modèle trouvé dans les chemins spécifiés.")
    return None

pipeline = load_model()
if pipeline is not None:
    print("✅ Pipeline de modèle chargé avec succès !")

# ⚠️ NOTE IMPORTANTE (limitation connue, à garder en tête) :
# Ce notebook charge le pipeline scikit-learn (.joblib) pour recalculer les
# prédictions de référence utilisées par Evidently. La production sert en
# réalité le modèle exporté en ONNX (ONNX Runtime), pas ce pipeline directement.
# Les deux devraient produire des résultats quasi identiques, mais un écart de
# précision numérique lié à l'export ONNX n'est pas totalement exclu.
# Piste d'amélioration future : scorer le jeu de référence via la session ONNX
# elle-même (comme le fait l'API en production) pour une comparaison rigoureuse
# "apples-to-apples" plutôt que via le pipeline scikit-learn d'origine.


📍 Chargement du modèle depuis : c:\Users\15GIRAV\Moi\Boulot\OpenClassrooms\P8_Confirmez_vos_competences_en_MLOPS_partie_2_2\models\best_pipeline_production.joblib
✅ Pipeline de modèle chargé avec succès !


## 3. Fonctions Utilitaires

In [18]:
def load_production_logs(log_path: Path) -> pd.DataFrame:
    """Lit le fichier predictions.jsonl et extrait les inputs ainsi que la prédiction."""
    if not log_path.exists():
        raise FileNotFoundError(f"Le fichier de log {log_path} n'existe pas.")

    records = []
    with open(log_path, mode="r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                if data.get("status") == "success":
                    entry = data["inputs"].copy()
                    entry["prediction"] = data.get("prediction")
                    records.append(entry)

    return pd.DataFrame(records)


def ensure_reference_predictions(reference_df: pd.DataFrame) -> pd.DataFrame:
    """Calcule les prédictions sur le jeu de référence si la colonne 'prediction' manque."""
    if "prediction" in reference_df.columns:
        return reference_df

    if pipeline is None:
        return reference_df

    df = reference_df.copy()
    for col in CATEGORICAL_FEATURES:
        if col in df.columns:
            df[col] = df[col].astype("category")

    cols_for_model = [c for c in df.columns if c in FEATURES]

    expected_order = getattr(
        getattr(pipeline, "named_steps", {}).get("imputer"), "feature_names_in_", None
    )
    if expected_order is not None:
        cols_for_model = list(expected_order)

    df["prediction"] = pipeline.predict(df[cols_for_model])
    return df


## 4. Démarrage du Run MLflow

On ouvre un run MLflow unique pour cette session de monitoring, dans lequel seront
loggués au fil des cellules suivantes : les métriques de santé opérationnelle, les
métriques et artefacts de drift, et les métriques de benchmark de performance.

In [19]:
# --- Démarrage du run MLflow pour cette session de monitoring ---
# On garde le run ouvert sur plusieurs cellules (santé opérationnelle, drift,
# perf API) pour que tout soit centralisé dans un seul enregistrement
# consultable et comparable dans l'UI MLflow (`mlflow ui`).
run = mlflow.start_run(run_name=f"monitoring_{OPTIMIZATION_PHASE_LABEL}_{RUN_TIMESTAMP}")

GIT_COMMIT = get_git_commit()
mlflow.set_tag("git_commit", GIT_COMMIT)
mlflow.set_tag("optimization_phase", OPTIMIZATION_PHASE_LABEL)
mlflow.set_tag("run_type", "monitoring_notebook")
mlflow.log_param("logs_file", str(LOGS_FILE))
mlflow.log_param("reference_data_path", str(REFERENCE_DATA_PATH))
mlflow.log_param("reference_data_hash", file_hash(REFERENCE_DATA_PATH))

print(f"🏃 Run MLflow démarré : {run.info.run_id}")
print(f"   Expérience : P8-MLOps-Monitoring | Phase : {OPTIMIZATION_PHASE_LABEL} | Commit : {GIT_COMMIT}")


Exception: Run with UUID 8250cd1c753440829f0f0eb00db22cc5 is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

## 5. Analyse de la Santé Opérationnelle de l'API

In [21]:
def load_production_logs(log_path: Path) -> tuple[pd.DataFrame, int]:
    """Lit le journal JSONL en ignorant les lignes corrompues (plutôt que de planter)."""
    if not log_path.exists():
        return pd.DataFrame(), 0
    records, n_bad = [], 0
    with open(log_path, mode="r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                n_bad += 1
    return pd.DataFrame(records), n_bad


df_logs, n_bad_lines = load_production_logs(LOGS_FILE)

if df_logs.empty:
    print(f"⚠️ Aucun log exploitable trouvé à : {LOGS_FILE}")
else:
    print("========================================")
    print("📈 MÉTRIQUES OPÉRATIONNELLES DE L'API")

    if n_bad_lines:
        print(f" • ⚠️ {n_bad_lines} lignes corrompues ignorées "
              f"(à corriger côté écriture API : 1 JSON compact par ligne)")

    total_reqs = len(df_logs)
    print(f" • Volume total de requêtes : {total_reqs}")
    mlflow.log_metric("health_total_requests", total_reqs)

    if "status" in df_logs.columns:
        success_rate = (df_logs["status"] == "success").mean() * 100
        print(f" • Taux de succès           : {success_rate:.2f}%")
        mlflow.log_metric("health_success_rate_pct", success_rate)
    else:
        print(" • ⚠️ Colonne 'status' absente — taux de succès non calculé")

    if "latency_ms" in df_logs.columns:
        lat = pd.to_numeric(df_logs["latency_ms"], errors="coerce").dropna()
        if not lat.empty:
            mean_lat, p95_lat, p99_lat = lat.mean(), lat.quantile(0.95), lat.quantile(0.99)
            print(f" • Latence moyenne         : {mean_lat:.2f} ms")
            print(f" • Latence P95             : {p95_lat:.2f} ms")
            print(f" • Latence P99             : {p99_lat:.2f} ms")
            mlflow.log_metric("health_latency_mean_ms", mean_lat)
            mlflow.log_metric("health_latency_p95_ms", p95_lat)
            mlflow.log_metric("health_latency_p99_ms", p99_lat)
    print("========================================")

📈 MÉTRIQUES OPÉRATIONNELLES DE L'API
 • ⚠️ 27 lignes corrompues ignorées (à corriger côté écriture API : 1 JSON compact par ligne)
 • Volume total de requêtes : 2
 • ⚠️ Colonne 'status' absente — taux de succès non calculé


In [22]:
with open(LOGS_FILE, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 8:
            break
        print(f"L{i+1}: {line[:120]!r}")

print("\n--- Lignes valides ---")
for rec in df_logs.to_dict("records"):
    print(rec)

L1: '{\n'
L2: '  "features_model": [\n'
L3: '    "customer_value_score",\n'
L4: '    "Panier_Moyen_N_signature_3",\n'
L5: '    "clp_contrat_ap_stat",\n'
L6: '    "GrandCompte",\n'
L7: '    "Turnover_N_signature_1",\n'
L8: '    "annees_depuis_dernier_achat",\n'

--- Lignes valides ---
{0: 'Famille_12_N_signature_2'}
{0: 'division'}


## 6. Détection du Data Drift & Target Drift (Evidently AI)

⚠️ **Limitation connue** : les prédictions de référence sont calculées avec le
pipeline scikit-learn (`.joblib`), alors que la production sert le modèle exporté en
ONNX. Un écart de précision numérique minime entre les deux n'est pas exclu — piste
d'amélioration listée dans la synthèse finale.

Le rapport HTML est désormais **horodaté** (au lieu d'écraser toujours le même
fichier) et **loggé en artefact MLflow**, avec le JSON complet du rapport en filet de
sécurité si le schéma Evidently change de version.

In [ ]:
print("🔄 Chargement et préparation des jeux de données...")

current_df = load_production_logs(LOGS_FILE)
print(f"📊 Logs de production chargés : {len(current_df)} enregistrements.")
mlflow.log_metric("drift_current_rows", len(current_df))

if REFERENCE_DATA_PATH.exists():
    reference_df = pd.read_csv(REFERENCE_DATA_PATH)
    reference_df = ensure_reference_predictions(reference_df)
else:
    print(f"⚠️ Fichier de référence introuvable ({REFERENCE_DATA_PATH}). Simulation par découpage des logs.")
    split_idx = int(len(current_df) * 0.5)
    reference_df = current_df.iloc[:split_idx].copy()
    current_df = current_df.iloc[split_idx:].copy()
    mlflow.set_tag("reference_data_source", "split_from_logs_fallback")

mlflow.log_metric("drift_reference_rows", len(reference_df))

cols_to_compare = [c for c in FEATURES if c in current_df.columns and c in reference_df.columns]

column_mapping = ColumnMapping()
column_mapping.numerical_features = [c for c in NUMERICAL_FEATURES if c in cols_to_compare]
column_mapping.categorical_features = [c for c in CATEGORICAL_FEATURES if c in cols_to_compare]

has_prediction = "prediction" in current_df.columns and "prediction" in reference_df.columns
if has_prediction:
    cols_to_compare.append("prediction")
    column_mapping.prediction = "prediction"

ref_subset = reference_df[cols_to_compare]
curr_subset = current_df[cols_to_compare]

# Génération du rapport Evidently
metrics = [DataDriftPreset()]
if has_prediction:
    metrics.append(TargetDriftPreset())

drift_report = Report(metrics=metrics)
drift_report.run(reference_data=ref_subset, current_data=curr_subset, column_mapping=column_mapping)

# --- Historisation : rapport HTML horodaté (au lieu d'écraser toujours le même fichier) ---
report_html_path = REPORTS_DIR / f"data_drift_report_{RUN_TIMESTAMP}.html"
drift_report.save_html(str(report_html_path))
print(f"✅ Rapport Data Drift sauvegardé dans : {report_html_path}")

# --- Historisation : JSON brut complet du rapport, loggé en artefact MLflow ---
# On garde le JSON intégral (plutôt que de ne parier que sur quelques clés) car
# le schéma exact du dictionnaire Evidently peut varier d'une version à l'autre.
report_dict = drift_report.as_dict()
report_json_path = REPORTS_DIR / f"data_drift_report_{RUN_TIMESTAMP}.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report_dict, f, default=str, ensure_ascii=False)

mlflow.log_artifact(str(report_html_path), artifact_path="drift_reports")
mlflow.log_artifact(str(report_json_path), artifact_path="drift_reports")

# --- Extraction "best effort" de quelques métriques scalaires pour comparaison directe entre runs ---
# Enveloppée dans un try/except : si le schéma Evidently change de version,
# le run n'échoue pas, on garde simplement le JSON complet en filet de sécurité.
try:
    dataset_drift_result = report_dict["metrics"][0]["result"]
    dataset_drift_detected = bool(dataset_drift_result.get("dataset_drift", False))
    drift_share = dataset_drift_result.get("drift_share")
    n_drifted_columns = dataset_drift_result.get("number_of_drifted_columns")

    mlflow.log_metric("drift_dataset_drift_detected", int(dataset_drift_detected))
    if drift_share is not None:
        mlflow.log_metric("drift_share", float(drift_share))
    if n_drifted_columns is not None:
        mlflow.log_metric("drift_n_drifted_columns", float(n_drifted_columns))

    print(f" • Dérive détectée (dataset_drift) : {dataset_drift_detected}")
    if drift_share is not None:
        print(f" • Part de colonnes en dérive      : {drift_share:.2%}")
except (KeyError, IndexError, TypeError) as e:
    mlflow.set_tag("drift_metric_extraction_warning", str(e))
    print(f"⚠️ Extraction des métriques scalaires de drift impossible ({e}). "
          f"Le rapport JSON complet reste disponible en artefact MLflow.")

# Affichage HTML interactif dans le notebook
display(HTML(filename=str(report_html_path)))


🔄 Chargement et préparation des jeux de données...


FileNotFoundError: Le fichier de log c:\Users\15GIRAV\Moi\Boulot\OpenClassrooms\P8_Confirmez_vos_competences_en_MLOPS_partie_2_2\logs\predictions.jsonl n'existe pas.

## 7. Benchmark de Performance de l'API (comparaison avant/après optimisation)

Cette section répond directement au premier objectif SMART du mentor
(*fonctions asynchrones, connexion async à la BDD*) : elle mesure la latence réelle de
l'API déployée, en séquentiel et en concurrent, et logue ces métriques dans le run
MLflow courant sous l'étiquette `OPTIMIZATION_PHASE_LABEL`. En relançant le notebook
avant puis après une optimisation (avec une étiquette différente à chaque fois), les
deux runs deviennent directement comparables dans MLflow.

In [ ]:
# --- Benchmark de performance de l'API déployée ---
# Objectif : mesurer la latence de /predict AVANT et APRÈS chaque optimisation
# (ex: passage à un vrai pool de connexions asynchrone), et tracer chaque
# mesure dans MLflow sous l'étiquette OPTIMIZATION_PHASE_LABEL définie plus haut.
# Exécute ce notebook une première fois avec OPTIMIZATION_PHASE_LABEL="avant_xxx"
# avant ta modification, puis une seconde fois avec "apres_xxx" après —
# les deux runs seront directement comparables dans l'UI MLflow.

import time
import asyncio
import httpx


API_BASE_URL = os.getenv("API_BASE_URL", "https://p8-oc-mlops-2-2.onrender.com")
N_SEQUENTIAL_REQUESTS = 20
N_CONCURRENT_REQUESTS = 30

_BASE_PAYLOAD = {
    "customer_value_score": 50.0,
    "Panier_Moyen_N_signature_3": 120.5,
    "GrandCompte": False,
    "annees_depuis_dernier_achat": 1.5,
    "Turnover_N_signature_1": 3500.0,
    "Panier_Moyen_N_signature_1": 150.0,
    "%EC": 12.5,
    "Nb_lignes_N_signature_1": 8.0,
    "Turnover_N_signature_3": 1500.0,
    "Famille_2_N_signature_2": 0.0,
    "Panier_Moyen_N_signature_2": 135.0,
    "act_val_cust_3M": True,
    "annees_depuis_1ere_facture": 4.2,
    "Famille_0_N_signature_1": 0.0,
    "Famille_2_N_signature_1": 0.0,
    "Famille_11_N_signature_1": 0.0,
    "Famille_14_N_signature_1": 0.0,
    "Famille_9_N_signature_3": 0.0,
}

# Champs dont le type a changé entre versions de l'API (string -> float/int).
# On garde les deux variantes possibles ici, et on choisit la bonne dynamiquement
# ci-dessous en interrogeant le schéma réellement déployé, plutôt que de figer
# une hypothèse qui casse au prochain déploiement.
_VARIABLE_FIELDS = {
    "clp_contrat_ap_stat": {"string": "STAT_01", "number": 1.0},
    "division": {"string": "DIV_A", "number": 0.0},
}


def build_sample_payload(base_url: str) -> dict:
    """Construit un payload de test compatible avec le schéma OpenAPI réellement
    exposé par l'API à l'instant présent (introspection via /openapi.json),
    au lieu de supposer un schéma figé en dur qui peut devenir obsolète après
    un déploiement (c'est exactement ce qui a causé les 422 précédents :
    l'API déployée attendait encore des strings pour clp_contrat_ap_stat/division
    alors que le payload envoyait des floats)."""
    payload = dict(_BASE_PAYLOAD)

    try:
        with httpx.Client(base_url=base_url, timeout=15.0, verify=False) as client:
            resp = client.get("/openapi.json")
            resp.raise_for_status()
            spec = resp.json()
        properties = spec["components"]["schemas"]["ClientData"]["properties"]

        for field, variants in _VARIABLE_FIELDS.items():
            field_schema = properties.get(field, {})
            candidates = field_schema.get("anyOf", [field_schema])
            is_string_type = any(c.get("type") == "string" for c in candidates)
            payload[field] = variants["string"] if is_string_type else variants["number"]

        print("✅ Payload de benchmark construit dynamiquement depuis /openapi.json (schéma actuellement déployé).")
    except Exception as e:
        # Fallback conservateur : on part sur les valeurs texte (schéma historique)
        # si le schéma OpenAPI est illisible pour une raison quelconque.
        for field, variants in _VARIABLE_FIELDS.items():
            payload[field] = variants["string"]
        print(f"⚠️ Impossible de lire /openapi.json ({e}). Repli sur un payload texte par défaut.")

    return payload


SAMPLE_PAYLOAD = build_sample_payload(API_BASE_URL)


def run_sequential_benchmark(n: int) -> list:
    latencies = []
    errors = []
    with httpx.Client(base_url=API_BASE_URL, timeout=30.0, verify=False) as client:
        client.post("/predict", json=SAMPLE_PAYLOAD)
        for _ in range(n):
            start = time.perf_counter()
            resp = client.post("/predict", json=SAMPLE_PAYLOAD)
            elapsed_ms = (time.perf_counter() - start) * 1000
            if resp.status_code == 200:
                latencies.append(elapsed_ms)
            else:
                errors.append((resp.status_code, resp.text[:200]))
    if errors:
        print(f"⚠️ {len(errors)}/{n} requêtes en échec. Exemple : {errors[0]}")
    return latencies


async def run_concurrent_benchmark(n: int) -> tuple:
    """Envoie n requêtes /predict en parallèle et retourne (latences_ms, temps_total_s)."""
    async with httpx.AsyncClient(base_url=API_BASE_URL, timeout=30.0, verify=False) as client:
        # Requête de chauffe
        await client.post("/predict", json=SAMPLE_PAYLOAD)

        async def timed_request():
            start = time.perf_counter()
            try:
                resp = await client.post("/predict", json=SAMPLE_PAYLOAD)
                elapsed_ms = (time.perf_counter() - start) * 1000
                if resp.status_code == 200:
                    return elapsed_ms, None
                else:
                    return None, (resp.status_code, resp.text[:200])
            except httpx.HTTPError as exc:
                return None, (type(exc).__name__, str(exc)[:200])

        overall_start = time.perf_counter()
        results = await asyncio.gather(*[timed_request() for _ in range(n)])
        overall_elapsed_s = time.perf_counter() - overall_start

    latencies = [lat for lat, err in results if lat is not None]
    errors = [err for lat, err in results if err is not None]
    if errors:
        print(f"⚠️ {len(errors)}/{n} requêtes concurrentes en échec. Exemple : {errors[0]}")
    return latencies, overall_elapsed_s


def summarize(latencies: list) -> dict:
    if not latencies:
        return {}
    s = pd.Series(latencies)
    return {
        "mean_ms": s.mean(),
        "p95_ms": s.quantile(0.95),
        "min_ms": s.min(),
        "max_ms": s.max(),
    }


print(f"🚀 Benchmark séquentiel ({N_SEQUENTIAL_REQUESTS} requêtes)...")
seq_latencies = run_sequential_benchmark(N_SEQUENTIAL_REQUESTS)
seq_stats = summarize(seq_latencies)

print(f"🚀 Benchmark concurrent ({N_CONCURRENT_REQUESTS} requêtes simultanées)...")
concurrent_latencies, total_concurrent_time_s = await run_concurrent_benchmark(N_CONCURRENT_REQUESTS)
conc_stats = summarize(concurrent_latencies)
throughput_rps = len(concurrent_latencies) / total_concurrent_time_s if total_concurrent_time_s > 0 else 0.0

print("========================================")
print("🚀 BENCHMARK DE PERFORMANCE API")
if seq_stats:
    print(f" • Séquentiel  - latence moyenne : {seq_stats['mean_ms']:.2f} ms | P95 : {seq_stats['p95_ms']:.2f} ms")
if conc_stats:
    print(f" • Concurrent  - latence moyenne : {conc_stats['mean_ms']:.2f} ms | P95 : {conc_stats['p95_ms']:.2f} ms")
    print(f" • Débit sous charge             : {throughput_rps:.2f} req/s")
if not seq_stats and not conc_stats:
    print(" • ⚠️ Aucune requête n'a abouti — vérifie les messages d'erreur ci-dessus (schéma, réseau, ou API indisponible).")
print("========================================")

mlflow.log_param("perf_n_sequential_requests", N_SEQUENTIAL_REQUESTS)
mlflow.log_param("perf_n_concurrent_requests", N_CONCURRENT_REQUESTS)

if seq_stats:
    mlflow.log_metric("perf_sequential_latency_mean_ms", seq_stats["mean_ms"])
    mlflow.log_metric("perf_sequential_latency_p95_ms", seq_stats["p95_ms"])
if conc_stats:
    mlflow.log_metric("perf_concurrent_latency_mean_ms", conc_stats["mean_ms"])
    mlflow.log_metric("perf_concurrent_latency_p95_ms", conc_stats["p95_ms"])
    mlflow.log_metric("perf_throughput_rps", throughput_rps)


✅ Payload de benchmark construit dynamiquement depuis /openapi.json (schéma actuellement déployé).
🚀 Benchmark séquentiel (20 requêtes)...
⚠️ 20/20 requêtes en échec. Exemple : (400, '{"detail":"Erreur lors de l\'inférence ONNX : could not convert string to float: \'STAT_01\'"}')
🚀 Benchmark concurrent (30 requêtes simultanées)...
⚠️ 30/30 requêtes concurrentes en échec. Exemple : (400, '{"detail":"Erreur lors de l\'inférence ONNX : could not convert string to float: \'STAT_01\'"}')
🚀 BENCHMARK DE PERFORMANCE API
 • ⚠️ Aucune requête n'a abouti — vérifie les messages d'erreur ci-dessus (schéma, réseau, ou API indisponible).


## 8. Synthèse MLOps, Diagnostic de Dérive & Clôture du Run

In [ ]:
# --- Synthèse et clôture propre du run MLflow ---
synthesis_notes = f"""
Bilan Diagnostic (run '{OPTIMIZATION_PHASE_LABEL}', commit {GIT_COMMIT}) :
- Covariate Shift (Data Drift) : voir métriques drift_dataset_drift_detected / drift_share loggées ci-dessus.
- Prediction Drift (Target Drift) : voir le rapport HTML complet en artefact MLflow (drift_reports/).

Stratégie MLOps de réponse au drift :
1. Phase 1 - Diagnostic Qualité : vérification de l'intégrité du pipeline ETL et absence d'anomalies de formatage.
2. Phase 2 - Qualification Métier : analyse des évolutions comportementales ou de la saisonnalité.
3. Phase 3 - Ré-entraînement Automatisé : fenêtrage glissant pour ré-entraîner XGBoost et re-validation
   (F1-score / AUC) avant mise en production.
"""

mlflow.log_text(synthesis_notes, "synthesis_notes.txt")
print(synthesis_notes)

mlflow.end_run()
print(f"🏁 Run MLflow clôturé : {run.info.run_id}")
print("👉 Pour comparer ce run aux précédents, lance dans un terminal à la racine du projet :")
print(f"     mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print("   puis ouvre http://localhost:5000 et filtre sur l'expérience 'P8-MLOps-Monitoring'.")



Bilan Diagnostic (run 'baseline', commit 22f2f82) :
- Covariate Shift (Data Drift) : voir métriques drift_dataset_drift_detected / drift_share loggées ci-dessus.
- Prediction Drift (Target Drift) : voir le rapport HTML complet en artefact MLflow (drift_reports/).

Stratégie MLOps de réponse au drift :
1. Phase 1 - Diagnostic Qualité : vérification de l'intégrité du pipeline ETL et absence d'anomalies de formatage.
2. Phase 2 - Qualification Métier : analyse des évolutions comportementales ou de la saisonnalité.
3. Phase 3 - Ré-entraînement Automatisé : fenêtrage glissant pour ré-entraîner XGBoost et re-validation
   (F1-score / AUC) avant mise en production.

🏁 Run MLflow clôturé : 9724c2c19997406abd9c7fa23ddfce5f
👉 Pour comparer ce run aux précédents, lance dans un terminal à la racine du projet :
     mlflow ui --backend-store-uri sqlite:///c:/Users/15GIRAV/Moi/Boulot/OpenClassrooms/P8_Confirmez_vos_competences_en_MLOPS_partie_2_2/mlflow.db
   puis ouvre http://localhost:5000 et fil

In [ ]:
# --- Export d'un tableau récapitulatif de l'historique MLflow (pour le rendu) ---
runs_df = mlflow.search_runs(experiment_names=["P8-MLOps-Monitoring"])

summary_cols = [
    "tags.optimization_phase", "tags.git_commit", "start_time",
    "metrics.health_latency_mean_ms", "metrics.health_latency_p95_ms",
    "metrics.perf_sequential_latency_mean_ms", "metrics.perf_concurrent_latency_p95_ms",
    "metrics.drift_share",
]
existing_cols = [c for c in summary_cols if c in runs_df.columns]
summary = runs_df[existing_cols].sort_values("start_time")

summary_path = REPORTS_DIR / "historique_optimisations.csv"
summary.to_csv(summary_path, index=False)
print(f"✅ Historique exporté : {summary_path}")
summary

✅ Historique exporté : c:\Users\15GIRAV\Moi\Boulot\OpenClassrooms\P8_Confirmez_vos_competences_en_MLOPS_partie_2_2\reports\historique_optimisations.csv


,tags.optimization_phase,tags.git_commit,start_time,metrics.health_latency_mean_ms,metrics.health_latency_p95_ms,metrics.drift_share
1,baseline,22f2f82,2026-08-31 14:09:54.016000+00:00,26.344523,74.974,0.5
0,baseline,22f2f82,2026-08-31 14:19:41.169000+00:00,26.344523,74.974,0.5


### Bilan Diagnostic (à mettre à jour après lecture des résultats ci-dessus)
* **Covariate Shift (Data Drift) :** à documenter à partir des métriques `drift_*` loggées et du rapport HTML en artefact.
* **Prediction Drift (Target Drift) :** idem, à partir du rapport complet.

### Stratégie MLOps de Réponse au Drift
1. **Phase 1 - Diagnostic Qualité :** vérification de l'intégrité du pipeline ETL et absence d'anomalies de formatage.
2. **Phase 2 - Qualification Métier :** analyse des évolutions comportementales ou de la saisonnalité.
3. **Phase 3 - Ré-entraînement Automatisé :** application d'un fenêtrage glissant pour ré-entraîner XGBoost et re-validation relative de la performance (F1-score / AUC) avant mise en production.

### Suivi de l'historique des optimisations (nouveau)
Toutes les exécutions de ce notebook sont désormais tracées dans l'expérience MLflow
`P8-MLOps-Monitoring` : métriques de santé, de drift et de performance API, rapport de
drift complet en artefact, et commit Git associé à chaque run. Pour visualiser et
comparer l'historique :

```bash
mlflow ui --backend-store-uri file:./mlruns
```

puis ouvrir `http://localhost:5000` et comparer les runs entre eux (vue tableau ou
graphique parallèle) pour objectiver l'impact de chaque phase d'optimisation.